# CareerOps-4B private GGUF export
Downloads private `telivity/CareerOps-4B-merged`, converts with llama.cpp, uploads Q4_K_M to private `telivity/CareerOps-4B-GGUF`.
**Do not flip public.**

In [ ]:
!pip -q install -U huggingface_hub "transformers>=4.51" sentencepiece protobuf

In [ ]:
import os
from pathlib import Path

auth = sorted(Path('/kaggle/input').glob('**/hf_token'))
assert auth, 'need careerops-runtime-auth'
os.environ['HF_TOKEN'] = Path(auth[0]).read_text().strip()
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
print('HF_TOKEN: present')

SRC = 'telivity/CareerOps-4B-merged'
OUT_REPO = 'telivity/CareerOps-4B-GGUF'
HF_DIR = Path('/kaggle/working/hf_merged')
GGUF_DIR = Path('/kaggle/working/gguf')
LLAMA = Path('/kaggle/working/llama.cpp')
HF_DIR.mkdir(parents=True, exist_ok=True)
GGUF_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from huggingface_hub import snapshot_download, HfApi, create_repo

print('downloading merged…', flush=True)
snapshot_download(SRC, local_dir=str(HF_DIR), token=os.environ['HF_TOKEN'])
print('downloaded', list(HF_DIR.iterdir())[:10], flush=True)

In [ ]:
import subprocess, sys

if not (LLAMA / 'convert_hf_to_gguf.py').exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', str(LLAMA)])

# Build quantize tool only (convert is pure python)
subprocess.check_call(['cmake', '-B', 'build', '-DGGML_NATIVE=OFF'], cwd=str(LLAMA))
subprocess.check_call(['cmake', '--build', 'build', '--config', 'Release', '-j', '2', '--target', 'llama-quantize'], cwd=str(LLAMA))

f16 = GGUF_DIR / 'CareerOps-4B-f16.gguf'
q4 = GGUF_DIR / 'CareerOps-4B-Q4_K_M.gguf'

print('converting HF → GGUF f16…', flush=True)
subprocess.check_call([
    sys.executable, str(LLAMA / 'convert_hf_to_gguf.py'),
    str(HF_DIR), '--outfile', str(f16), '--outtype', 'f16'
])
print('quantize → Q4_K_M…', flush=True)
quantize = LLAMA / 'build' / 'bin' / 'llama-quantize'
if not quantize.exists():
    quantize = LLAMA / 'build' / 'bin' / 'Release' / 'llama-quantize'
subprocess.check_call([str(quantize), str(f16), str(q4), 'Q4_K_M'])
print('sizes', f16.stat().st_size, q4.stat().st_size, flush=True)

In [ ]:
from huggingface_hub import HfApi, create_repo

tok = os.environ['HF_TOKEN']
create_repo(OUT_REPO, private=True, exist_ok=True, token=tok)
api = HfApi(token=tok)
api.upload_file(path_or_fileobj=str(q4), path_in_repo=q4.name, repo_id=OUT_REPO, token=tok,
                commit_message='private Q4_K_M GGUF from CareerOps-4B-merged')
# also upload a tiny README
readme = Path('/kaggle/working/GGUF_README.md')
readme.write_text(
    '# CareerOps-4B GGUF (private)\n\n'
    'Q4_K_M from private merged `telivity/CareerOps-4B-merged`.\n'
    'Do not treat as public until explicit approval.\n'
)
api.upload_file(path_or_fileobj=str(readme), path_in_repo='README.md', repo_id=OUT_REPO, token=tok,
                commit_message='private GGUF readme')
info = api.repo_info(OUT_REPO, repo_type='model')
assert info.private is True
print('uploaded PRIVATE', OUT_REPO, flush=True)